# 第19周 · AI Agents：工具调用、ReAct 循环与评测（Capstone 消融与初稿）

**本周结束时你能做到**：分清单次 LLM 调用、workflow（预定义步骤）、工具调用 Agent（由模型决定下一步）与自主研究 Agent；按"名称、描述、参数、输出"写出工具定义；从零写一个约 100 行的 ReAct 循环——模型提出动作 → 执行工具 → 把观察结果追加进上下文 → 判断是否结束；为同一任务再写一个固定步骤的 workflow；在 20 道有标准答案的题上比较两者的准确率、调用次数与 token；用五个类别做失败分析；说清楚为什么"调通一次 API"不等于"做了一项 AI 研究"。

本周的四个蓝本：

- Anthropic，[*Building Effective Agents*](https://www.anthropic.com/research/building-effective-agents)（Erik Schluntz & Barry Zhang，2024-12-19）：workflow 与 agent 的区分、五种 workflow 模式、"先找最简单的方案"；
- Yao et al., *ReAct: Synergizing Reasoning and Acting in Language Models*, ICLR 2023（[arXiv:2210.03629](https://arxiv.org/abs/2210.03629)）：Thought → Action → Observation 循环；
- Hugging Face [Agents Course · Unit 1](https://huggingface.co/learn/agents-course/unit1/introduction)（Apache-2.0）：Thought–Action–Observation 循环与工具的组成。课程**不**引入 smolagents、LangGraph 等框架——框架会把提示词和控制流藏起来，而本周的目的恰恰是看清它们；
- Claude Messages API 的[工具调用文档](https://platform.claude.com/docs/en/agents-and-tools/tool-use/overview)。

> **两种后端，同一份代码。** 配置了 API 密钥（见下文）时，下面的代码调用真实的 Claude（`claude-opus-5-5`，每次调用都计费）；没有密钥时，调用工具包里的 `MockLLM`——一个**照剧本演戏**的假模型，返回的对象和 Claude SDK 的一模一样。真实 API 会拒绝的请求，它同样拒绝（抛出的异常就是 `anthropic.BadRequestError` 的子类）；真实 API 接受、但不推荐的写法，它照样接受，只打印一次提醒。CI 和没有 API 预算时走后者，零成本跑通整本 notebook。记住：**MockLLM 的准确率是剧本写的，不代表任何真实模型的能力**；它如实反映的只有控制流、消息格式，以及 token 随对话增长的结构。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")
# 本周新增依赖：Anthropic 官方 Python SDK（只有配置了 API 密钥、调用真实 Claude 时才用到；
# 没有密钥时，课程用工具包里的 MockLLM 跑通全部代码）
if importlib.util.find_spec("anthropic") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic>=2.3,<3"], check=True)
print("anthropic SDK 已就绪")

In [ ]:
import json
import re
import time
import anthropic
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from zero2researcher import agents
from zero2researcher.checks import week19 as ck
from zero2researcher.checks import report

MODEL = agents.MODEL                     # "claude-opus-5-5"
THINKING = {"type": "adaptive", "display": "summarized"}   # thinking 块返回可读的推理摘要（默认 "omitted" 时文字为空）
items = agents.load_eval()               # 20 道评测题
item_by_id = {it["id"]: it for it in items}
print("模型:", MODEL, "| 评测题:", len(items))

### 后端：真实 Claude 还是 MockLLM

`agents.make_backend()` 这样决定：环境变量里有 `ANTHROPIC_API_KEY`（在 Colab 上会先尝试从 Secrets 里读取）并且装了 `anthropic` → 返回真实的 `anthropic.Anthropic().messages`；否则 → 返回 `MockLLM`。两者都只有一个方法 `create(...)`，参数和返回值的结构完全相同。

想在 Colab 上用自己的密钥：点左侧栏的钥匙图标（Secrets）→ 新增一个名为 `ANTHROPIC_API_KEY` 的 Secret → 打开本 notebook 的 "Notebook access" 开关 → 重新运行下一格。**永远不要把密钥写进 notebook、不要 `print` 它、不要提交到 GitHub**；分享 notebook 之前先"清除所有输出"。

In [ ]:
llm = agents.make_backend()

---
## S1 · 从单次调用到 Agent

### 四个层级

*Building Effective Agents* 把各种"LLM + 工具"的系统统称为 agentic systems，再按**谁来决定下一步**分成两类：**workflow** 用预先写好的代码路径来编排 LLM 和工具；**agent** 则由 LLM 动态地决定自己的流程和工具使用，自己掌控怎样完成任务。加上两端，本周用四个层级来思考：

| 层级 | 谁决定下一步 | 例子 | 本周的代码 |
| --- | --- | --- | --- |
| 单次 LLM 调用 | 没有"下一步"：一问一答 | 把一段摘要翻译成中文；给一条影评分类 | `run_direct`（基线） |
| Workflow（预定义代码路径） | **你写的代码** | 先判断题型 → 查资料 → 列算式 → 作答 | `run_workflow` |
| 工具调用 Agent | **模型**：每一步自己决定调哪个工具、何时结束 | ReAct | `run_agent` |
| 自主研究 Agent | 模型；任务开放、步数多，要自己规划、写代码、跑实验 | 文献调研 Agent；自动跑消融的"AI 科学家"原型 | 不实现，S4 讨论 |

文中总结了五种常见的 workflow 模式（下表是转述）：

| 模式 | 做法 | 适合 |
| --- | --- | --- |
| Prompt chaining（提示链） | 把任务拆成固定的几步，上一步的输出是下一步的输入，中间可以插入程序检查 | 步骤清楚、能事先写死的任务 |
| Routing（路由） | 先给输入分类，再交给专门的后续流程 | 输入有几类明显不同的情况 |
| Parallelization（并行） | 把任务切成独立的几块并行处理（sectioning），或同一问题问几次再投票（voting） | 子任务互不依赖；想要更可靠的判断 |
| Orchestrator-workers（编排者–工人） | 一个中心 LLM 动态拆分子任务、分派给工人 LLM、再汇总 | 子任务无法事先确定 |
| Evaluator-optimizer（评估–优化） | 一个 LLM 生成，另一个按标准评估并给反馈，循环改进 | 有清晰的评价标准、迭代确实有用 |

文章最核心的建议是：**先找最简单的方案，只在确实需要时才增加复杂度**——很多应用一次调用加上检索和几个好例子就够了；agentic 系统用更高的延迟和成本换取更好的任务表现（值不值得，要看任务），Agent 的错误还会在多步之间累积。本周的 workflow 就是 routing + prompt chaining；它和 Agent 谁更好，不靠争论，靠下面的评测。

### 编程 Agent 与研究型 Agent

- **编程 Agent**（例如 Claude Code、GitHub Copilot 的 agent 模式）：读代码 → 改文件 → 跑测试 → 看报错 → 再改。它的工具是"读写文件、执行命令"，观察是测试输出——**测试本身就是现成的评测集**，这是这类 Agent 进步最快的原因之一。
- **研究型 Agent**（各家的 deep research 功能，以及把实验也自动化的"AI 科学家"类原型）：检索文献、读论文、写代码跑实验、汇总成报告。难点在于没有现成的测试告诉它"做对了没有"——**怎样评测它，本身就是一个研究问题**。这正是 S4 要说的：给 Agent 建评测集、设对照、做失败分析，才是研究。

### 工具 = 名称 + 描述 + 参数 + 输出

HF Agents Course 列出一个工具包含的几项：一段说明它做什么的文字描述、一个可调用对象（Python 函数）、带类型的参数，以及（可选的）带类型的输出；工具名出现在它据此生成的工具说明里。模型看不到你的 Python 代码，**它只能读到这份说明书**，并据此决定什么时候调用、传什么参数。Claude API 的工具定义就是一个 dict：

- `name`：工具名；
- `description`：做什么、接受什么、返回什么、什么时候该用——写得越清楚，模型越少用错；
- `input_schema`：一个 JSON Schema，描述参数（类型、说明、哪些必填）；`"additionalProperties": False` 禁止模型编造 schema 之外的参数。

输出写在描述里（"return its numeric value as text"）。跟着写一遍计算器的定义：

In [ ]:
# 在这里写你的代码
raise NotImplementedError
ck.tool_schema(calculator_tool)
print("与工具包里的定义相同：", calculator_tool == agents.CALCULATOR_TOOL)
print(json.dumps(agents.WIKI_SEARCH_TOOL, indent=2))

工具包提供这三个工具的 Python 实现（`agents.TOOL_FUNCTIONS`，工具名 → 函数）：

- `calculator(expression)`：**安全**求值——用 `ast` 把表达式解析成语法树，只允许数字、`+ - * / **` 和括号，逐个节点计算。**绝不能对模型给出的字符串直接 `eval`**：模型的输出可能被它检索到的内容操纵（提示词注入），`eval` 等于把你的电脑交给了它；
- `wiki_search(query)`：在离线维基语料上做关键词检索，返回最相关的至多 3 篇。语料是 34 篇英文维基百科条目（大多取自导语）的简短改写，其中 12 篇支撑评测题，其余是关键词相近的干扰条目（BERT 与 Transformer、Pierre 与 Marie Curie……）。它是维基百科文本的改编作品，与原文一样按 [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/) 发布（署名：维基百科贡献者；每篇的 `url` 指向原文与作者记录）；
- `wiki_lookup(title)`：按确切标题取一篇。

还有一个联网版 `agents.live_wiki_summary(title, user_agent=...)`，调用维基百科的 REST 摘要接口。维基百科要求请求带一个描述性的 User-Agent：**不带、或用 Python 默认的 User-Agent，一律返回 403**。工具包的默认值只是一个通用的课程标识（`Zero2ResearcherCourse/1.0 (educational use)`，不含任何个人信息）；按维基媒体的 [User-Agent 政策](https://meta.wikimedia.org/wiki/User-Agent_policy)，你自己联网时应换成含**你本人联系方式**的字符串（项目网址或邮箱）。这个函数让连续请求至少间隔 1 秒，遇到 429（请求太多）时按 `Retry-After` 等待并最多重试 3 次。默认用离线版——CI 不联网，离线语料也让每个人的结果都能复现。

In [ ]:
print(agents.calculator("2 * (3 + 4)"), "|", agents.calculator("277 / 13"))
for bad in ["__import__('os').system('ls')", "2 ^ 10", "1 / 0", "radius * 2"]:
    try:
        agents.calculator(bad)
    except ValueError as e:
        print(f"{bad!r:34} → ValueError: {e}")
print()
print(agents.wiki_search("Marie Curie Nobel Prize"))

In [ ]:
USE_LIVE_WIKI = False     # 想试联网版就改成 True，并按维基媒体的 User-Agent 政策把 user_agent 换成含你自己联系方式的字符串
if USE_LIVE_WIKI:
    print(agents.live_wiki_summary("Marie Curie",
                                   user_agent="Zero2ResearcherStudent/0.1 (your-email@example.com)"))

### 一次工具调用的完整往返

先不写循环，手动走一遍。拿评测集里的一道题：Marie Curie 两次获得诺贝尔奖，中间隔了几年？把题目、系统提示和三个工具一起发给模型：

In [ ]:
item = item_by_id["f05"]
print(item["question"])
print("系统提示:", agents.AGENT_SYSTEM[:120], "...")
messages = [{"role": "user", "content": item["question"]}]
response = llm.create(model=MODEL, max_tokens=16000, system=agents.AGENT_SYSTEM, thinking=THINKING,
                      tools=agents.TOOLS, messages=messages)
print("\nstop_reason:", response.stop_reason,
      "| usage:", response.usage.input_tokens, "输入 /", response.usage.output_tokens, "输出 token")
for block in response.content:
    if block.type == "thinking":
        print("  thinking :", block.thinking or "（文字为空）")
    elif block.type == "text":
        print("  text     :", block.text)
    elif block.type == "tool_use":
        print("  tool_use :", block.name, block.input, "| id =", block.id)
    else:
        print("  " + block.type)

读懂这个回复：

- `stop_reason == "tool_use"`：模型还没说完，它在**请你执行工具**。其他取值：`end_turn`（说完了）、`max_tokens`（被截断）、`refusal`（拒答，详情在 `response.stop_details`）。**先看 stop_reason，再读 content**；
- `content` 是一串**块**：
  - `thinking` 块：Opus 5.5 的思考始终是自适应模式、不能关闭，回复**可以**以 thinking 块开头（简单的请求、低 effort 时也可能完全不思考）。默认（`display: "omitted"`）块里的文字是空的；我们在请求里加了 `thinking=THINKING`，即 `{"type": "adaptive", "display": "summarized"}`，它就返回一段可读的推理摘要；
  - `text` 块：系统提示要求模型每次调用工具前先用一句话说明打算做什么、为什么——这就是 ReAct 里的 **Thought**。注意：Opus 5.5 写在工具调用之间、超过一两句的说明，会放进 thinking 块而不是 text 块，所以 Thought 可能出现在这两种块里；
  - `tool_use` 块：**Action**——`name` + `input`（`input` 已经是 dict），`id` 用来和结果配对。

  所以永远**遍历块、检查 `.type`**，不要假设 `content[0]` 就是文字；
- `usage`：这一次调用的输入 / 输出 token 数，算成本全靠它。

执行工具，把结果作为 Observation 送回去。两条规则：assistant 消息**原样**追加，包括 thinking 块（删掉 thinking 块不会报错，但模型会丢掉之前的推理；**改写**之前任何一轮的内容，才会让 thinking 块失效，见 S3）；这一轮的**所有** `tool_result` 放进**同一条** user 消息，每个的 `tool_use_id` 等于对应 `tool_use` 块的 `id`：

In [ ]:
for _ in range(4):                                         # 最多再走几步；正式的循环在 S2 写
    messages.append({"role": "assistant", "content": response.content})          # 原样追加
    if response.stop_reason != "tool_use":
        break
    # 这里故意先不处理工具出错：真实模型若查一个语料里没有的标题，下一行会抛 ValueError——S2 的 run_tool 会补上
    results = [{"type": "tool_result", "tool_use_id": block.id,
                "content": agents.TOOL_FUNCTIONS[block.name](**block.input)}
               for block in response.content if block.type == "tool_use"]
    messages.append({"role": "user", "content": results})                        # 全部结果放进同一条消息
    response = llm.create(model=MODEL, max_tokens=16000, system=agents.AGENT_SYSTEM, thinking=THINKING,
                          tools=agents.TOOLS, messages=messages)

agents.print_transcript(messages)
final_text = "".join(b.text for b in response.content if b.type == "text")
print("\n最后一次 stop_reason:", response.stop_reason, "| 判分:", agents.grade(item, final_text))

传了 API 不接受的参数会怎样？以 `temperature` 为例（API 版本 2026-10-21 起，任何模型都不再接受采样参数）。要分清两层：

- **SDK 这一层**：anthropic SDK 2.x 的 `create()` 已经没有 `temperature` 参数，你一传，**本地**就抛 `TypeError`，请求根本发不出去；
- **API 这一层**：只有绕过 SDK（经 `extra_body`，或用旧版 SDK）把它塞进请求，API 才返回 HTTP 400，SDK 把它抛成 `anthropic.BadRequestError`。

MockLLM 两层都照样模拟，而且它的 400 错误（`agents.MockBadRequestError`）就是 `anthropic.BadRequestError` 的子类，所以同一个 `except` 对两种后端都适用：

In [ ]:
question_only = [{"role": "user", "content": item["question"]}]
try:
    llm.create(model=MODEL, max_tokens=100, temperature=0, messages=question_only)
except TypeError as e:                       # 真实 SDK 与 MockLLM 都在本地抛 TypeError，请求没有发出
    print("TypeError →", e)
try:
    llm.create(model=MODEL, max_tokens=100, extra_body={"temperature": 0}, messages=question_only)
except anthropic.BadRequestError as e:       # 绕过 SDK 发出去：API 返回 HTTP 400（不计费）
    print(f"{type(e).__name__}（HTTP {e.status_code}）→", str(e)[:150])

**真实 API 会直接拒绝（HTTP 400）的**还有：强制 `tool_choice`（`{"type": "any"}` 或指定工具，Opus 5.5 不支持——用默认的 `auto`，在提示词里说明该用哪个工具）、关闭思考或设 `budget_tokens`、以 assistant 消息结尾的"预填"、`tool_use` 没有配上 `tool_result`、`tool_result` 没有排在那条 user 消息的最前面，以及**改写已经发出的历史**（S3 细讲）。

**真实 API 接受、但有代价的**：同一轮的结果拆成几条相邻的 user 消息（API 会把相邻的同角色消息合并成一轮，不报错，但这会让模型以后更少并行调用）；回传 assistant 消息时删掉 thinking 块（不报错，但模型丢掉了之前的推理）。MockLLM 对这两种写法也照样接受，只打印一次 `MockLLMWarning` 提醒你这不是推荐写法。

这些规则写在 S2 的循环里，就不会再犯。

---
## S2 · 从零写 ReAct 循环

ReAct 的循环只有三拍：**Thought**（想下一步做什么）→ **Action**（调用一个工具）→ **Observation**（读工具返回的结果），重复直到给出答案。在知识密集型任务（多跳问答 HotpotQA 与事实核查 FEVER）上，原文的动作空间只有三个：`search[entity]`（返回该条目页面的前 5 句；条目不存在时返回维基搜索给出的 5 个相似条目）、`lookup[string]`（在当前页面里找下一句含该字符串的句子，相当于浏览器里的 Ctrl+F）、`finish[answer]`（结束）。另外两个决策类基准 ALFWorld 与 WebShop 用的是环境自己的动作。对应到我们的实现——**工具名与 ReAct 的动作名只是相似，语义并不一一对应**：

| ReAct | 本周的实现 |
| --- | --- |
| Thought | `text` 块（系统提示要求每次调用工具前用一句话说明）；Opus 5.5 上较长的中间说明在 `thinking` 块里，要设 `display: "summarized"` 才看得到文字 |
| `search[entity]` | `wiki_lookup(title)`：按确切标题取整篇摘要，相当于条目存在时的 `search`；`wiki_search(query)`：关键词检索、返回至多 3 篇，相当于"返回相似条目"那一支 |
| `lookup[string]`（页内查找） | 本课**没有**对应的工具：离线摘要只有两三句，不需要在页内查找 |
| （原文没有） | `calculator(expression)` |
| Observation | 我们执行工具后追加的 `tool_result` 块 |
| `finish[answer]` | `stop_reason == "end_turn"`，回复最后一行 `Final answer: ...` |

原文有两个值得记住的发现。其一，单独用 ReAct 在 HotpotQA 上的精确匹配率（27.4）反而**低于**只做思维链推理、不查资料的 CoT（29.4），两者结合才最好——**Agent 不会自动更好**。其二，作者人工检查 ReAct 的失败样本，约 29% 属于"标签歧义"：答案其实对了，只是和标准答案字面不一致——**判分规则本身就会制造失败**。这两点分别对应今天的评测对照和答案规范化。

先把"执行一个 `tool_use` 块"写成函数。工具出错（表达式不合法、标题不存在、工具名不认识）是正常情况，不能让程序崩：把错误信息作为 `is_error: True` 的结果还给模型，让它自己调整。

In [ ]:
def run_tool(block):
    """执行一个 tool_use 块，返回对应的 tool_result 块（dict）。"""
    # 在这里写你的代码
    raise NotImplementedError


ck.tool_result_block(run_tool)

现在是循环本身。每一轮四件事：调用模型 → 把回复原样追加 → 如果 `stop_reason` 不是 `"tool_use"` 就结束 → 否则执行全部工具、把结果放进同一条 user 消息，进入下一轮。

结束时要分清**怎样结束的**，记进 `status`：只有 `end_turn` 才取文字作答案（`"answered"`）；`refusal`（拒答，content 可能为空，也可能只剩半句话）记 `"refused"`，`max_tokens`（回复被截断）记 `"truncated"`，这两种都**没有答案**（answer 保持 None），也不执行工具——被截断的回复里，tool_use 可能只写了一半。外面再套一个 **`max_steps` 保护**：模型可能陷入"一直调工具"的循环，没有上限的 `while True` 会把预算烧光；用完步数记 `"max_steps"`。

顺便记账：调用次数、工具次数、token，以及每次工具调用的记录 `trace`（失败分析要用）——每条记录里也存下这一轮的 Thought（thinking 摘要 + text）。

In [ ]:
def new_result():
    """一次运行的记录：答案、调用次数、工具次数、token、工具调用轨迹，以及怎样结束的（status）。"""
    return {"answer": None, "llm_calls": 0, "tool_calls": 0, "input_tokens": 0, "output_tokens": 0,
            "trace": [], "stop_reason": None, "status": None}


STATUS = {"end_turn": "answered", "refusal": "refused", "max_tokens": "truncated"}
"""stop_reason → 这次运行怎样结束；用完 max_steps 记为 "max_steps"。"""


def run_agent(question, llm, max_steps=8, system=agents.AGENT_SYSTEM):
    """ReAct 循环。返回 new_result() 的字段，外加完整的对话历史 messages。"""
    res = new_result()
    messages = [{"role": "user", "content": question}]
    res["messages"] = messages
    for step in range(max_steps):
        # 在这里写你的代码
        raise NotImplementedError
    res["status"] = "max_steps"     # 用完 max_steps 仍没有结束：answer 保持 None
    return res


ck.react_loop(run_agent)

In [ ]:
out = run_agent(item_by_id["f06"]["question"], llm)
agents.print_transcript(out["messages"])
print({k: out[k] for k in ("llm_calls", "tool_calls", "input_tokens", "output_tokens", "stop_reason", "status")})
print("判分:", agents.grade(item_by_id["f06"], out["answer"]))

注意第一轮：模型在**一个回复里发出两个 `wiki_search`**（两个年份互不依赖，可以并行查），我们把两个结果放进同一条 user 消息。如果拆成两条相邻的 user 消息，真实 API 会把它们合并成一轮、不会报错，但这会让模型"学到"不该并行调用（官方 Parallel tool use 文档）；MockLLM 也接受这种写法，只打印一次提醒，`ck.react_loop` 则判它不通过。

### 同一任务的固定步骤 workflow

现在把控制权从模型手里拿回来。workflow 版本是 routing + prompt chaining：

```
route（LOOKUP 还是 ARITHMETIC）
  └─ LOOKUP → query（≤ 2 个检索词）→ wiki_search × ≤ 2 → 记进笔记
expression（写一个算式，或 NONE）→ calculator → 记进笔记
answer（只依据笔记作答）
```

每一步都是一次**不带工具**的普通调用，用 `agents.WORKFLOW_PROMPTS` 里对应的提示词作 system——**调哪个工具、调几次，由你的代码决定**，模型只负责填空。下面先给出两个小助手：`ask` 调一次模型并记账，`call_tool` 执行一个工具并记进 trace。

In [ ]:
for step, prompt in agents.WORKFLOW_PROMPTS.items():
    print(f"{step:10s} | {prompt}")

In [ ]:
def ask(llm, step, user_text, res):
    """workflow 的一步：以 WORKFLOW_PROMPTS[step] 为 system 调用一次模型（不带工具），返回回复文字。"""
    response = llm.create(model=MODEL, max_tokens=16000, system=agents.WORKFLOW_PROMPTS[step],
                          messages=[{"role": "user", "content": user_text}])
    res["llm_calls"] += 1
    res["input_tokens"] += response.usage.input_tokens
    res["output_tokens"] += response.usage.output_tokens
    res["stop_reason"] = response.stop_reason
    if response.stop_reason != "end_turn":           # 拒答或被截断：这一步没有可用的输出
        res["status"] = res["status"] or STATUS.get(response.stop_reason, response.stop_reason)
        return ""
    return "".join(b.text for b in response.content if b.type == "text").strip()


def call_tool(name, args, res):
    """由代码（而不是模型）决定调用的工具：执行、记进 trace，返回输出文字（出错时返回错误信息）。"""
    try:
        output, is_error = str(agents.TOOL_FUNCTIONS[name](**args)), False
    except Exception as e:
        output, is_error = f"Error: {e}", True
    res["trace"].append({"name": name, "input": args, "output": output, "is_error": is_error})
    res["tool_calls"] += 1
    return output


def run_workflow(question, llm):
    """固定步骤：route →（query → wiki_search）→ expression → calculator → answer。"""
    res = new_result()
    notes = []
    # 在这里写你的代码
    raise NotImplementedError
    return res


def run_direct(question, llm):
    """单次调用基线：不给工具，一问一答。"""
    res = new_result()
    response = llm.create(model=MODEL, max_tokens=16000, system=agents.DIRECT_SYSTEM,
                          messages=[{"role": "user", "content": question}])
    res["llm_calls"] = 1
    res["input_tokens"], res["output_tokens"] = response.usage.input_tokens, response.usage.output_tokens
    res["stop_reason"] = response.stop_reason
    res["status"] = STATUS.get(response.stop_reason, response.stop_reason)
    if response.stop_reason == "end_turn":
        res["answer"] = "".join(b.text for b in response.content if b.type == "text") or None
    return res


ck.workflow(run_workflow)
out = run_workflow(item_by_id["f07"]["question"], llm)
for t in out["trace"]:
    print(f"{t['name']}({t['input']}) → {t['output'][:90]}")
print(out["answer"], "| LLM 调用", out["llm_calls"], "次")

### 在 20 道题上评测

评测集 `agents.load_eval()`：10 道多步算术应用题（每题需要 2–3 次计算，含百分比、单位换算、"只算整袋"这类陷阱）+ 10 道事实题（答案在离线维基语料里，事实都能在维基百科上核对；其中 4 道要先查到年份再相减；题面明确问"两个年份之差"，避免和"实际经过了多少年"混淆）。每题记录了 `gold`（标准答案）、`source`（依据的维基条目）和 `evidence`（一次正确的运行应在工具输出里见到的信息）。

**判分**用 `agents.grade(item, answer)`：先取回答里最后一个 `Final answer:` 之后的部分，再按答案类型比较——数字按数值比（去掉千位分隔逗号，`1,953.`、`378.0`、`15.3%` 都能对上；`Final answer:` 后面只能有一个数——写出两个不同的数就是"两头下注"，判错），文字按规范化后比（小写、去标点、去冠词）。看几个例子，体会"直接比字符串"有多脆弱：

In [ ]:
cases = [("Final answer: 1953", 1953), ("Final answer: 1,953.", 1953), ("The answer is 378.0", 378),
         ("Final answer: 15.3%", 15.3), ("FINAL ANSWER: alexander fleming.", "Alexander Fleming"),
         ("Final answer: twenty-one", 21), ("Final answer: about 21.3, so 21 full bags", 21),
         ("Final answer: 1953, 1954", 1953)]
pd.DataFrame([{"model output": p, "gold": g,
               "naive string ==": p.split(":")[-1].strip() == str(g),
               "agents.is_correct": agents.is_correct(p, g)} for p, g in cases])

规范化救回了中间四个；接下来两个"内容对了、形式不对"，它救不回来——这一类失败在 S3 叫"格式不匹配"。最后一个是"两头下注"：同时写出两个年份，只要其中一个对就算对的话，模型多写几个候选就能"蒙对"，所以判错。规范化规则必须**在看结果之前**定好，并对所有系统一视同仁，否则就是在给自己喜欢的系统放水。

三个系统各跑一遍 20 道题：

In [ ]:
def evaluate(system_fn, items, llm):
    """在全部评测题上跑一个系统，每题一条记录。"""
    records = []
    for item in items:
        try:
            out = system_fn(item["question"], llm)
        except (anthropic.APIConnectionError, anthropic.APIStatusError) as e:
            if isinstance(e, anthropic.APIStatusError) and e.status_code < 500 and e.status_code != 429:
                raise                    # 400 之类说明代码有错：不要吞掉，先修代码
            out = dict(new_result(), status="error", error=f"{type(e).__name__}: {e}")   # 重试后仍失败：记一条，继续
        records.append({"id": item["id"], "type": item["type"], "answer": out["answer"],
                        "correct": agents.grade(item, out["answer"]),
                        "status": out["status"], "stop_reason": out["stop_reason"], "error": out.get("error"),
                        "llm_calls": out["llm_calls"], "tool_calls": out["tool_calls"],
                        "input_tokens": out["input_tokens"], "output_tokens": out["output_tokens"],
                        "trace": out["trace"], "messages": out.get("messages")})
    return records


SYSTEMS = {"direct": run_direct, "workflow": run_workflow, "agent": run_agent}
t0 = time.time()
results = {name: evaluate(fn, items, llm) for name, fn in SYSTEMS.items()}
print(f"3 个系统 × {len(items)} 题，用时 {time.time() - t0:.1f} 秒")

summary = pd.DataFrame({
    name: {"accuracy": np.mean([r["correct"] for r in recs]),
           "mean_llm_calls": np.mean([r["llm_calls"] for r in recs]),
           "mean_tool_calls": np.mean([r["tool_calls"] for r in recs]),
           "mean_input_tokens": np.mean([r["input_tokens"] for r in recs]),
           "mean_output_tokens": np.mean([r["output_tokens"] for r in recs])}
    for name, recs in results.items()}).T
ck.eval_table(results, summary)
summary.round(2)

In [ ]:
by_type = pd.DataFrame({name: pd.DataFrame(recs).groupby("type")["correct"].mean()
                        for name, recs in results.items()}).T
tokens = summary[["mean_input_tokens", "mean_output_tokens"]]
acc, tok = summary["accuracy"], tokens.sum(axis=1)


def best(question_type):
    col = by_type[question_type]
    winners = [s for s in col.index if np.isclose(col[s], col.max())]
    if len(winners) == len(col):
        return f"all tied at {col.max():.0%}"
    return " and ".join(winners) + (" tied" if len(winners) > 1 else "")


fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
by_type.plot.bar(ax=axes[0], rot=0)
axes[0].set_ylim(0, 1.25)
axes[0].set_xlabel("System")
axes[0].set_ylabel("Accuracy (20 questions, 1 run)")
axes[0].set_title(f"Arithmetic: {best('arithmetic')} best; lookup: {best('lookup')}")
axes[0].legend(title="Question type", ncol=2, loc="upper center")
tokens.plot.bar(stacked=True, ax=axes[1], rot=0, color=["C2", "C3"])
axes[1].set_xlabel("System")
axes[1].set_ylabel("Tokens per question")
axes[1].set_title(f"The agent spends {tok['agent'] / tok['workflow']:.1f}x the workflow's tokens")
axes[1].legend(["Input", "Output"])
fig.suptitle(f"Agent {acc['agent']:.0%} vs workflow {acc['workflow']:.0%} correct; "
             f"no-tool baseline {acc['direct']:.0%}")
plt.tight_layout()
plt.show()

怎样读这张图（**数字取决于后端**：mock 下由剧本决定；只有真实 Claude 下才是测量）：

- **成本的差别是结构性的**，mock 也如实反映：Agent 每走一步，都把系统提示、三个工具的定义（API 还会为工具额外注入约 300 token 的系统提示）和**越来越长的对话历史**整个重发一遍；workflow 每次调用只带当前这一步需要的几行。
- **准确率的差别要靠对照和重复**：同一个模型、同一套工具、同一套题、同一个判分规则，只改变"谁来决定下一步"。真实模型不能固定 temperature，每次运行的结果可能不同——作业里每题跑 3 次，报告均值 ± 标准差；只跑一次就下"A 比 B 好"的结论，和只用一个随机种子比较两个网络一样不可靠。
- **看基线**：单次调用不查资料，在事实题上照样可能全对——这些都是著名的事实，强模型早已背下。基线已经接近满分的题，区分不了两个系统；想测"真检索"，评测集里就要有模型不可能背下来的事实（例如最新的数据、你自己实验室的记录）。**设计评测集本身就是研究的一部分。**

---
## S3 · 记忆、上下文与 RAG；失败分析

### 上下文就是模型的全部"记忆"

模型本身不记得上一次调用。每次 `create(...)`，它能看到的只有这一次请求里的内容：系统提示 + 工具定义 + `messages`。所以 Agent 的"短期记忆"就是 `messages` 列表——每走一步，整段历史重发一遍，输入 token 随步数增长：

In [ ]:
demo = agents.MockLLM()                  # 这里固定用 mock：只看结构，不花钱
run_agent(item_by_id["a03"]["question"], demo)
steps = range(1, len(demo.log) + 1)
fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.plot(steps, [e["input_tokens"] for e in demo.log], "o-", label="Input tokens")
ax.plot(steps, [e["output_tokens"] for e in demo.log], "s--", label="Output tokens")
ax.plot(steps, np.cumsum([e["input_tokens"] for e in demo.log]), "^:", label="Cumulative input tokens")
ax.set_xticks(list(steps))
ax.set_xlabel("Agent step (LLM call)")
ax.set_ylabel("Tokens (simulated)")
ax.set_title("Each step re-sends the whole history, so total cost grows faster than the steps")
ax.legend()
plt.tight_layout()
plt.show()

- **短期记忆 = 上下文窗口。** 窗口再大也有上限，而且越长越贵、越慢，模型在超长的上下文里也更容易漏看信息。长任务的常见做法：清掉旧的工具结果、把早期对话**压缩成摘要**、只保留关键结论。**但在 Opus 5.5 上，不要在客户端自己改写已经发出去的历史**：历史必须只追加——改动、删除或缩短之前的任何一轮（包括把旧的 tool_result 改成"已清理"）、中途改 system 或增删工具，都会让之后的 thinking 块失效，2026-08-31 及以后创建的账号会直接收到 HTTP 400（`Invalid signature in thinking block ... bound to a different conversation`）。清理和压缩要交给服务器端：**context editing**（`client.beta.messages.create(..., betas=["context-management-2025-06-27"], context_management={"edits": [{"type": "clear_tool_uses_20250919"}]})`，清掉旧的工具结果）或 **compaction**（beta `compact-2026-01-12`，接近上限时自动把早期对话总结成摘要）——它们不算改写历史。MockLLM 也按这条规则检查，见下一格。
- **长期记忆 = 外部存储。** 跨会话要记住的东西（用户偏好、已完成的实验、读过的论文笔记）存进文件或数据库，需要时再检索出来放进上下文。编程 Agent 常把项目约定写进一个说明文件、每次启动时读入——这就是最朴素的长期记忆。
- **RAG（检索增强生成）**：先检索，再把检索到的内容放进上下文，最后生成（Lewis et al., *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*, NeurIPS 2020，[arXiv:2005.11401](https://arxiv.org/abs/2005.11401)）。我们的 workflow 就是一个最小的 RAG：`query → wiki_search → 笔记 → answer`；Agent 版由模型自己决定"查不查、查什么、查几次"，常被称为 agentic RAG。RAG 的上限由检索决定——检索不到，生成得再好也没用；失败分析里"检索词查不到所需条目"就是它的典型症状。

In [ ]:
# 演示：在客户端"清掉旧的工具结果"再接着发请求——MockLLM 与真实 API（新账号）一样返回 400
demo = agents.MockLLM()
history = run_agent(item_by_id["a03"]["question"], demo, max_steps=2)["messages"]   # 走两步，停在一个 tool_use 之后
if len(history) >= 5:
    edited = list(history)
    edited[2] = {"role": "user", "content": [dict(r, content="(cleared)") for r in history[2]["content"]]}
    try:
        demo.create(model=MODEL, max_tokens=16000, system=agents.AGENT_SYSTEM, thinking=THINKING,
                    tools=agents.TOOLS, messages=edited)
    except anthropic.BadRequestError as e:
        print(f"{type(e).__name__}（HTTP {e.status_code}）→", str(e)[:200])
    ok = demo.create(model=MODEL, max_tokens=16000, system=agents.AGENT_SYSTEM, thinking=THINKING,
                     tools=agents.TOOLS, messages=history)                            # 原样的历史：照常继续
    print("不改写历史：stop_reason =", ok.stop_reason)
else:
    print("先完成 S2 的 run_agent，再运行这一格")

### 失败分析：五个类别

只看准确率，你只知道"错了几道"；读轨迹，才知道"为什么错、该改什么"。

**先看这次运行怎样结束**（`status`）。拒答（`"refused"`）、被截断（`"truncated"`）、用完步数（`"max_steps"`，例如一直重复同一个调用——ReAct 原文把"无法从重复步骤中恢复"归入推理错误）都不属于下面五类里的任何一类，**单独计数**：拒答要看 `stop_details` 里的类别，考虑改写请求或换模型；截断要加大 `max_tokens`；超步数要看模型是不是陷入了循环。

其余的失败，每条**只标一个主因**：

| 类别 | 定义 | 在轨迹里看到什么 | 常见对策 |
| --- | --- | --- | --- |
| 工具选择错误 | 调用了不该用的工具；workflow 里路由走错了分支 | 算术题去查维基；拿一句话去问计算器；查资料题被路由成算术题 | 改进工具描述；在系统提示里说明何时用哪个工具；改进路由 |
| 参数错误 | 工具选对了，但传进去的参数错了 | 算式列错（"花掉 30%"写成 `× 0.30`）；单位没换算；检索词查不到所需条目 | 参数描述里给例子；用 `strict` 保证格式；把多步计算拆开 |
| 提前结束 | 还没拿到所需信息（或没做完所需的计算）就给出答案 | 一次工具都没调就作答；查到两个年份却不做减法 | 系统提示要求"先查后答"；在 workflow 里把必需的步骤写死 |
| 幻觉 / 编造 | 观察结果里已有正确信息，答案却与之矛盾，或凭空捏造 | 检索结果写着 2017，答案却是 2018 | 要求答案引用观察结果；加一步程序化核对 |
| 格式不匹配 | 答案内容其实对了，只是判分器没认出来 | `twenty-one`、`about 21.3, so 21`、`299 792 458` | 在提示里规定答案格式；改进规范化（**在看结果之前定好、对所有系统一视同仁**） |

判断顺序（第一条成立的就是主因）：① 内容对、形式错 → 格式不匹配；② 用错了工具 → 工具选择错误；③ 工具对、参数错 → 参数错误；④ 没拿到所需信息就作答 → 提前结束；⑤ 有信息却答错 → 幻觉 / 编造。这五类是把 ReAct 原文的失败分类（推理错误、检索结果无用、幻觉、标签歧义）细化到"工具调用"场景。

下面用 MockLLM 的**故障注入**把每一类各演示一次——`MockLLM(faults={题号: 故障})` 只在这道题上注入这一种故障：

In [ ]:
DEMOS = [("a10", "wrong_tool"), ("a07", "bad_args"), ("f07", "early_stop"),
         ("f08", "hallucination"), ("f09", "format_mismatch")]
demo_runs = {}
for qid, fault in DEMOS:
    item = item_by_id[qid]
    out = run_agent(item["question"], agents.MockLLM(faults={qid: fault}))
    demo_runs[qid] = out
    print(f"===== {qid} · 注入 {fault}（{agents.FAULT_CATEGORY[fault]}）· 标准答案 {item['gold']} · "
          f"判分 {agents.grade(item, out['answer'])}")
    agents.print_transcript(out["messages"])
    print()

STOP_DEMOS = [("a01", "loop"), ("f05", "refusal"), ("a05", "truncated")]   # 三种非正常结束
for qid, fault in STOP_DEMOS:
    out = run_agent(item_by_id[qid]["question"], agents.MockLLM(faults={qid: fault}))
    demo_runs[qid] = out
    print(f"{qid} · 注入 {fault}（{agents.STOP_CATEGORY[fault]}）→ status={out['status']!r}, "
          f"stop_reason={out['stop_reason']!r}, answer={out['answer']!r}, LLM 调用 {out['llm_calls']} 次")

几十条轨迹逐条读之前，可以先用一个启发式分类器**初筛**：先按 `status` 把拒答 / 截断 / 超步数单独拣出来，其余的按上面的判断顺序，借助每题的 `evidence` 字段检查工具输出里有没有出现应有的信息。它只是初筛——规则有盲区（例如"只查了两个年份中的一个"会被它判成参数错误），最后要靠人读轨迹核对。

In [ ]:
def numbers_in(text):
    text = re.sub(r"(?<=\d)[ ,](?=\d{3}(?!\d))", "", str(text))          # 去掉千位分隔（逗号或空格）
    return [float(x) for x in re.findall(r"(?<![\w.])-?\d+(?:\.\d+)?", text)]


def seen(text, value):
    """value 是否出现在 text 里：数字按数值比较，文字按规范化后的子串比较。"""
    if re.fullmatch(r"-?[\d,]+(?:\.\d+)?", str(value)):
        v = float(str(value).replace(",", ""))
        return any(abs(x - v) <= 1e-6 * max(1.0, abs(v)) for x in numbers_in(text))
    return agents.normalize_answer(value) in agents.normalize_answer(text)


STOP_LABELS = {"refused": "拒答", "truncated": "截断", "max_steps": "超步数", "error": "API 错误"}


def triage(item, record, system):
    """启发式初筛：非正常结束单独归类；其余按判断顺序返回五个类别之一。答对时返回 None。"""
    if record["correct"]:
        return None
    if record.get("status") in STOP_LABELS:           # 拒答 / 截断 / 超步数 / API 错误：不进入五类
        return STOP_LABELS[record["status"]]
    answer, trace = record["answer"] or "", record["trace"]
    used = [t["name"] for t in trace]
    outputs = "\n".join(str(t["output"]) for t in trace)
    if any(seen(answer, g) for g in [item["gold"]] + item.get("aliases", [])):
        return "格式不匹配"
    wiki_used = any(name.startswith("wiki") for name in used)
    if item["type"] == "arithmetic" and wiki_used:
        return "工具选择错误"
    if item["type"] == "lookup" and not wiki_used and (used or system == "workflow"):
        return "工具选择错误"                 # agent 拿计算器查事实；workflow 路由走错了分支
    if not trace:
        return "提前结束"
    if not all(seen(outputs, e) for e in item["evidence"]):
        return "参数错误"
    if item["type"] == "lookup" and item["needs_calculation"] and not seen(outputs, item["gold"]):
        return "提前结束"                     # 查到了数据，却跳过了计算
    return "幻觉/编造"


LABELS = {**agents.FAULT_CATEGORY, **agents.STOP_CATEGORY}
for qid, fault in DEMOS + STOP_DEMOS:
    rec = dict(demo_runs[qid], correct=agents.grade(item_by_id[qid], demo_runs[qid]["answer"]))
    print(f"{qid}: 注入「{LABELS[fault]}」→ 初筛「{triage(item_by_id[qid], rec, 'agent')}」")
report("初筛结果与注入的故障一致",
       all(triage(item_by_id[q], dict(demo_runs[q], correct=False), "agent") == LABELS[f]
           for q, f in DEMOS + STOP_DEMOS))

把初筛用在 S2 的评测结果上，得到一张"错题清单"：

In [ ]:
rows = []
for system in ("agent", "workflow"):
    for rec in results[system]:
        if not rec["correct"]:
            item = item_by_id[rec["id"]]
            rows.append({"system": system, "id": rec["id"], "gold": item["gold"], "status": rec["status"],
                         "answer (last line)": ((rec["answer"] or "").strip().splitlines() or [""])[-1],
                         "tools used": [t["name"] for t in rec["trace"]],
                         "triage": triage(item, rec, system)})
if isinstance(llm, agents.MockLLM) and llm.faults == "demo":
    expected = {(s, q): agents.FAULT_CATEGORY[f] for s, d in agents.DEMO_FAULTS.items() for q, f in d.items()}
    report("初筛结果与 mock 注入的故障一致", {(r["system"], r["id"]): r["triage"] for r in rows} == expected)
pd.DataFrame(rows)

从这张表能读出三个具体的方向（以 mock 为例；换成真实 Claude 后，请用你自己的表）：

- workflow 让模型**无法跳过检索步骤**（前提是路由正确）：查资料题一定会先检索。但这不等于排除了提前结束——expression 一步写出 NONE，它照样会"没算就答"；这类情形在初筛里多半被判成"工具选择错误"，是因为判断顺序里路由错误排在前面。同时它**引入了**新的失败：路由一旦判错，后面整条链都走在错误的分支上（工具选择错误）；计算器没有取整，workflow 又只允许写一个算式，需要中途取整的题（a06：先算出能做 18 个整蛋糕，再乘单价）就没法在算式中间取整——模型只能先心算出 18 再写 `18 * 18`，剧本里它写了一个不取整的算式（参数错误）。
- Agent 的失败更多样：可以不查就答（提前结束）、查了却不信（幻觉）、拿错工具——**灵活性的代价是更多的出错方式**。
- 格式不匹配两边都有：不是模型"不会"，而是我们没把格式说清楚。对策（在系统提示里给出答案格式的例子）是一个**可检验的假设**：改提示 → 在同一套题上重跑 → 比较格式不匹配的数量，其余一切不变。注意 mock 的剧本不会因为你改了提示词而改变，这类实验要在真实模型上做。

Anthropic 的工程博客 [*Demystifying evals for AI agents*](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents)（2026-01）给出的建议与此一致：从 20–50 个来自真实失败的任务起步，每个任务多次试验，并且——**读轨迹**。

---
## S4 · Capstone 工作时段：消融、结果图与初稿

### 简单 API 调用不等于 AI research

回头看今天写的代码：真正"调用模型"的只有一行 `llm.create(...)`。如果这就是全部，它是一次演示，不是一项研究。今天的工作之所以算研究，是因为另外四件事：

1. **评测集**：20 道题都有标准答案、依据（`source`）和判分规则；规则在看结果之前定好，对所有系统一视同仁。
2. **对照**：同一个模型、同一套工具、同一套题，只改变一个变量——"谁来决定下一步"（单次调用 / workflow / Agent）。没有对照，"Agent 答对了 15 道"毫无意义：可能不用工具也能答对 15 道。
3. **重复与统计**：真实模型不能设 temperature，同一题每次运行的结果可能不同；每题 ≥ 3 次，报告均值 ± 标准差，差异小于波动时不下结论。
4. **失败分析**：逐条读轨迹、归类、提出**可检验**的改进假设，再用同一套评测去验证。

这四件事和你的 Capstone 完全同构：评测集 = 你的测试集与指标；对照 = 基线与主方法；重复 = 多种子；失败分析 = 消融与失败案例。第 20 周写最终报告时，读者最先追问的就是这四个问题。

### 本周 Capstone 里程碑：消融实验与报告初稿

S3 余下的时间与 S4 是 Capstone 工作时段。本周要完成：

- **消融实验**：从完整方法出发，一次只拿掉（或替换）一个组件，其余设置完全相同——同一组种子（≥ 3）、同一划分、同一训练预算。把结果填进"设置 | 均值 ± 标准差 | 相对完整方法的变化 | 是否大于一个标准差"的表。
- **结果图**：主结果图 + 消融图。轴标签、单位、误差棒，标题写结论。
- **报告初稿**：按 `templates/capstone_report.md`（4 页简化会议论文格式：Abstract / Introduction / Related Work / Method / Experiments / Limitations，外加 AI 助手使用声明与 Reproducibility）写出初稿；至少 Method 与 Experiments 两节要完整。
- **互查**（约 30 分钟；有学习伙伴就交换图表与初稿，没有的话隔一天后自己当审稿人重读，也可以让 AI 助手照这份清单审）：
  - [ ] 不看讲解，能读懂每一张图吗？（轴、单位、误差棒、结论式标题）
  - [ ] 每个数字都是 ≥ 3 个种子的均值 ± 标准差吗？
  - [ ] 基线是否和主方法一样认真调过参？
  - [ ] 消融是否一次只动一个组件？
  - [ ] 结论有没有超出证据（例如把单一数据集上的结果写成普遍规律）？
  - [ ] 只凭 Method 一节，你能重做这个实验吗？
  - [ ] 这份初稿最可能被审稿人追问的一个问题是什么？写下来。

### 本周小结

- 单次调用 → workflow（代码决定下一步）→ 工具调用 Agent（模型决定下一步）→ 自主研究 Agent；先找最简单的方案，只在需要时增加复杂度
- 工具定义 = 名称 + 描述 + 参数（`input_schema`，`additionalProperties: False`）+ 输出；计算器绝不 `eval`
- ReAct 循环：调用 → 原样追加回复 → 看 `stop_reason` → 执行全部 `tool_use` → 所有 `tool_result` 放进同一条 user 消息 → 重复，外加 `max_steps` 保护；记下怎样结束的（answered / refused / truncated / max_steps）
- 历史只追加：不在客户端改写已发出的消息；清理上下文用服务器端的 context editing 或 compaction
- 评测：有标准答案的题、事先定好的规范化判分、只改一个变量的对照；真实模型每题 ≥ 3 次
- 上下文就是模型的全部记忆，每一步都整段重发；RAG = 检索 + 放进上下文 + 生成
- 失败分析：拒答、截断、超步数单独计数；其余分五类——工具选择错误、参数错误、提前结束、幻觉 / 编造、格式不匹配——读轨迹，每条只标一个主因
- 调通 API 是演示；评测集 + 对照 + 重复 + 失败分析才是研究